# 07 — Iterative Backtest Notebook
Standalone experiment notebook. It rebuilds the same Silver/Gold feature
engineering as Pipe-1 from the raw source tables, trains the same two-stage
LightGBM model as `03_train_model.py`, and evaluates validation/internal
test/final inference in **iterative** mode:
- the scored horizon never sees its true `quantite`;
- week `t+1` uses the model prediction from week `t` to rebuild `lag_1`,
rolling stats, trends, zero-rates, expanding stats, etc.;
- validation and internal test are therefore block-forecast backtests.


%pip install lightgbm==4.3.0
dbutils.library.restartPython()


In [ ]:
import math
import sys
sys.path.append("./")

import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
from mlflow.models import infer_signature
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, LongType, StringType, StructField, StructType
from pyspark.sql.window import Window



# -----------------------------------------------------------------------------
# Autonomous configuration: raw tables, splits, features, model params
# -----------------------------------------------------------------------------
NOM_EQUIPE = "telecacaton"
TABLE_PREDICTIONS = f"workspace.default.predictions_equipe_{NOM_EQUIPE}"

TBL_TRAIN = "workspace.default.histo_ventes_train"
TBL_TEST = "workspace.default.histo_ventes_test"
TBL_AGENCE = "workspace.default.donnees_agence"
TBL_ARTICLES = "workspace.default.donnees_articles"
TBL_FACTURATION = "workspace.default.donnees_facturation"

PAIR_KEYS = ["code_agence", "code_article"]

TRAIN_END_WEEK_ID = 202426
VAL_START_WEEK_ID = 202427
VAL_END_WEEK_ID = 202452
INTERNAL_TEST_START_WEEK_ID = 202501
INTERNAL_TEST_END_WEEK_ID = 202526
FINAL_INFERENCE_START_WEEK_ID = 202527
FINAL_INFERENCE_END_WEEK_ID = 202552

SEED = 42
OUTLIER_PERCENTILE = 0.995
ANOMALY_MULTIPLIER = 10.0
ANOMALY_ROLL_WINDOW = 26

LAGS_ALL = [1, 2, 4, 8, 13, 26, 52, 104]
ROLLING_WINDOWS = [4, 8, 13, 26, 52]
ROLLING_MEDIAN_WINDOWS = [4, 13]

FEATURES_NUMERIC = [
    "lag_1", "lag_2", "lag_4", "lag_8", "lag_13", "lag_26", "lag_52", "lag_104",
    "roll_mean_4", "roll_mean_8", "roll_mean_13", "roll_mean_26", "roll_mean_52",
    "roll_std_4", "roll_std_8", "roll_std_13", "roll_std_26", "roll_std_52",
    "roll_median_4", "roll_median_13",
    "zero_rate_26", "zero_rate_52", "pair_zero_rate_expanding",
    "trend_8", "ratio_n1_vs_mean", "yoy_ratio",
    "pair_mean", "pair_median", "pair_max", "pair_count", "pair_cv",
    "sem_mean", "sem_max", "sem_median",
    "agence_mean", "agence_median",
    "article_mean", "article_median",
    "n_active_weeks",
    "fac_prix_unit", "fac_pct_pro", "fac_nb_chantiers", "fac_nb_achats",
    "annee", "num_sem", "sin_sem", "cos_sem",
    "is_summer_trough", "is_xmas_trough",
]
FEATURES_CATEGORICAL = [
    "art_specialite_enc",
    "art_famille_enc",
    "art_marque_enc",
    "art_mdd_enc",
    "ag_region_enc",
]
FEATURES = FEATURES_NUMERIC + FEATURES_CATEGORICAL

LGB_PARAMS_ZERO = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "num_leaves": 127,
    "min_child_samples": 50,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "scale_pos_weight": 1.5,
    "n_jobs": -1,
    "seed": SEED,
    "verbose": -1,
}
LGB_NUM_ROUNDS_ZERO = 1800

LGB_PARAMS_QTY = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.5,
    "metric": "None",
    "learning_rate": 0.03,
    "num_leaves": 255,
    "min_child_samples": 30,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "max_bin": 511,
    "n_jobs": -1,
    "seed": SEED,
    "verbose": -1,
}
LGB_NUM_ROUNDS_QTY = 2500

MLFLOW_EXPERIMENT = f"/Shared/sgdb2026_{NOM_EQUIPE}_iterative_notebook"


# -----------------------------------------------------------------------------
# Autonomous utility functions
# -----------------------------------------------------------------------------
EPS = 1e-10


def wape_numpy(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + EPS))


def wape_lgb_feval(y_pred, dataset):
    y_true = dataset.get_label()
    return "wape", wape_numpy(y_true, y_pred), False


mlflow.set_experiment(MLFLOW_EXPERIMENT)



def materialize_table(df, table_name):
    """Serverless-friendly replacement for cache/persist."""
    if MATERIALIZE_INTERMEDIATE_DELTA:
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(table_name)
        )
        return spark.table(table_name)
    return df


## 0. Controls


In [ ]:
# Iterative validation model selection is the expensive part. Coordinate search
# keeps the protocol leakage-free while avoiding a full cartesian grid by default.
RUN_ITERATIVE_MODEL_SELECTION = True
MODEL_SELECTION_STRATEGY = "coordinate"  # "coordinate" or "exhaustive"
MODEL_SELECTION_COORDINATE_PASSES = 1
DEFAULT_ZERO_THRESHOLD = 0.65
THRESHOLD_GRID_MODEL_SELECT = [0.55, 0.65, 0.75]

# Used only when RUN_ITERATIVE_MODEL_SELECTION = False.
MANUAL_ZERO_ITER = None
MANUAL_QTY_ITER = None
MANUAL_ZERO_THRESHOLD = 0.65

MATERIALIZE_INTERMEDIATE_DELTA = True
# Fast iterative model selection controls. Expand these after debugging.
ZERO_ITER_CANDIDATE_STEP = 300
QTY_ITER_CANDIDATE_STEP = 300
MAX_ZERO_ITER_CANDIDATES = 5
MAX_QTY_ITER_CANDIDATES = 6
ITER_FEED_ROUNDED = True
TMP_PREFIX = f"workspace.default.tmp_iter_nb_{NOM_EQUIPE}"
TMP_SILVER = f"{TMP_PREFIX}_silver_ventes"
TMP_PANEL = f"{TMP_PREFIX}_panel"
TMP_TRAIN_FEATURES = f"{TMP_PREFIX}_train_features"

OUT_VAL_ITER = "workspace.default.iterative_nb_val_predictions"
OUT_INTERNAL_TEST_ITER = "workspace.default.iterative_nb_internal_test_predictions"
OUT_FINAL_ITER = f"workspace.default.iterative_nb_predictions_equipe_{NOM_EQUIPE}"

print(f"Train           : <= {TRAIN_END_WEEK_ID}")
print(f"Validation      : {VAL_START_WEEK_ID}..{VAL_END_WEEK_ID}")
print(f"Internal test   : {INTERNAL_TEST_START_WEEK_ID}..{INTERNAL_TEST_END_WEEK_ID}")
print(f"Final inference : {FINAL_INFERENCE_START_WEEK_ID}..{FINAL_INFERENCE_END_WEEK_ID}")
print(f"Iterative model selection: {RUN_ITERATIVE_MODEL_SELECTION} / {MODEL_SELECTION_STRATEGY}")
print(f"Iter threshold grid: {THRESHOLD_GRID_MODEL_SELECT}")


## 1. Pipe-1 Silver Reconstruction


In [ ]:
def _add_time_columns(df, semaine_col="semaine"):
    return (
        df.withColumn("annee", F.split(F.col(semaine_col), "-").getItem(0).cast("int"))
          .withColumn("num_sem", F.split(F.col(semaine_col), "-").getItem(1).cast("int"))
          .withColumn("week_id", F.col("annee") * F.lit(100) + F.col("num_sem"))
    )


def _encode_column(df, src, dst):
    if src not in df.columns:
        return df.withColumn(dst, F.lit(-1).cast("int"))
    dim = (
        df.select(src)
        .distinct()
        .withColumn(dst, F.dense_rank().over(Window.orderBy(src)) - F.lit(1))
    )
    return df.join(dim, src, "left")


def build_silver_ventes(train_raw):
    raw = (
        train_raw
        .transform(_add_time_columns)
        .withColumnRenamed("quantite", "quantite_raw")
    )

    pair_stats = (
        raw.groupBy(*PAIR_KEYS)
        .agg(
            F.sum("quantite_raw").alias("_pair_sum"),
            F.expr("percentile_approx(quantite_raw, 0.995)").alias("_pair_p995"),
            F.expr("percentile_approx(quantite_raw, 0.5)").alias("_pair_median"),
        )
        .withColumn("is_dead_pair", (F.col("_pair_sum") == 0).cast("tinyint"))
    )

    capped = (
        raw.join(pair_stats, PAIR_KEYS, "left")
        .withColumn(
            "_cap_value",
            F.when(F.col("is_dead_pair") == 1, F.lit(None))
             .otherwise(F.greatest(F.col("_pair_p995"), F.col("_pair_median") * F.lit(2.0))),
        )
        .withColumn(
            "is_capped",
            (
                F.col("_cap_value").isNotNull()
                & (F.col("quantite_raw") > F.col("_cap_value"))
            ).cast("tinyint"),
        )
        .withColumn(
            "quantite_capped",
            F.when(F.col("is_capped") == 1, F.col("_cap_value").cast("double"))
             .otherwise(F.col("quantite_raw").cast("double")),
        )
    )

    roll_w = (
        Window.partitionBy(*PAIR_KEYS)
        .orderBy("week_id")
        .rowsBetween(-ANOMALY_ROLL_WINDOW, -1)
    )
    with_roll = (
        capped
        .withColumn("_roll_median", F.expr("percentile_approx(quantite_capped, 0.5)").over(roll_w))
        .withColumn(
            "is_anomaly",
            (
                F.col("_roll_median").isNotNull()
                & (F.col("_roll_median") > F.lit(0))
                & (F.col("quantite_capped") > F.lit(ANOMALY_MULTIPLIER) * F.col("_roll_median"))
            ).cast("tinyint"),
        )
        .withColumn(
            "quantite",
            F.when(F.col("is_anomaly") == 1, F.col("_roll_median"))
             .otherwise(F.col("quantite_capped"))
             .cast("long"),
        )
    )

    smooth_w = (
        Window.partitionBy(*PAIR_KEYS)
        .orderBy("week_id")
        .rowsBetween(-13, -1)
    )

    return (
        with_roll
        .withColumn("quantite_smooth", F.avg("quantite").over(smooth_w))
        .select(
            "semaine", "annee", "num_sem", "week_id",
            "code_agence", "code_article",
            F.col("quantite_raw").cast("long").alias("quantite_raw"),
            F.col("quantite").cast("long").alias("quantite"),
            F.col("quantite_smooth").cast("double").alias("quantite_smooth"),
            F.col("is_anomaly").cast("tinyint"),
            F.col("is_capped").cast("tinyint"),
            F.col("is_dead_pair").cast("tinyint"),
        )
    )


def build_articles_encoded(articles_raw):
    df = articles_raw
    mapping = [
        ("specialite", "art_specialite_enc"),
        ("famille", "art_famille_enc"),
        ("marque", "art_marque_enc"),
        ("article_mdd", "art_mdd_enc"),
    ]
    for src, dst in mapping:
        df = _encode_column(df, src, dst)

    keep = ["code_agence", "code_article"] + [dst for _, dst in mapping]
    return df.select(*keep).dropDuplicates(["code_agence", "code_article"])


def build_agences_encoded(agences_raw):
    src = "region" if "region" in agences_raw.columns else "ag_region"
    df = agences_raw.withColumnRenamed(src, "ag_region")
    df = _encode_column(df, "ag_region", "ag_region_enc")
    return df.select("code_agence", "ag_region_enc").dropDuplicates(["code_agence"])


def build_facturation_lagged(fac_raw):
    def _pick(*candidates, default=None):
        for c in candidates:
            if c in fac_raw.columns:
                return F.col(c)
        return F.lit(default)

    year_col = _pick("annee", "year")
    month_col = _pick("mois", "month")

    monthly = (
        fac_raw
        .withColumn("_annee", year_col.cast("int"))
        .withColumn("_mois", month_col.cast("int"))
        .groupBy("code_agence", "code_article", "_annee", "_mois")
        .agg(
            F.sum(_pick("sum_montant", default=0.0)).alias("_sum_montant"),
            F.sum(_pick("sum_quantite", default=0.0)).alias("_sum_quantite"),
            F.sum(_pick("nb_achats", default=0.0)).alias("fac_nb_achats"),
            F.sum(_pick("nb_achats_par_professionnels", default=0.0)).alias("_nb_pro"),
            F.sum(_pick("nb_chantiers", default=0.0)).alias("fac_nb_chantiers"),
        )
        .withColumn(
            "fac_prix_unit",
            F.when((F.col("_sum_quantite").isNull()) | (F.col("_sum_quantite") == 0), None)
             .otherwise(F.col("_sum_montant") / F.col("_sum_quantite")),
        )
        .withColumn(
            "fac_pct_pro",
            F.when((F.col("fac_nb_achats").isNull()) | (F.col("fac_nb_achats") == 0), None)
             .otherwise(F.col("_nb_pro") / F.col("fac_nb_achats")),
        )
    )

    return (
        monthly
        .withColumn("_shifted_mois", F.col("_mois") + F.lit(2))
        .withColumn(
            "_join_annee",
            F.when(F.col("_shifted_mois") > F.lit(12), F.col("_annee") + F.lit(1))
             .otherwise(F.col("_annee")),
        )
        .withColumn(
            "_join_mois",
            F.when(F.col("_shifted_mois") > F.lit(12), F.col("_shifted_mois") - F.lit(12))
             .otherwise(F.col("_shifted_mois")),
        )
        .select(
            "code_agence", "code_article", "_join_annee", "_join_mois",
            "fac_prix_unit", "fac_pct_pro",
            F.col("fac_nb_chantiers").cast("double"),
            F.col("fac_nb_achats").cast("double"),
        )
    )


def build_silver_panel(cleaned, test_raw):
    test = (
        test_raw
        .transform(_add_time_columns)
        .withColumn("quantite_raw", F.lit(None).cast("long"))
        .withColumn("quantite", F.lit(None).cast("long"))
        .withColumn("quantite_smooth", F.lit(None).cast("double"))
        .withColumn("is_anomaly", F.lit(0).cast("tinyint"))
        .withColumn("is_capped", F.lit(0).cast("tinyint"))
        .withColumn("is_dead_pair", F.lit(0).cast("tinyint"))
        .select(*cleaned.columns)
    )

    pair_w = Window.partitionBy(*PAIR_KEYS)
    return cleaned.unionByName(test).withColumn(
        "is_dead_pair",
        F.max("is_dead_pair").over(pair_w).cast("tinyint"),
    )


In [ ]:
train_raw = spark.table(TBL_TRAIN)
test_raw = spark.table(TBL_TEST)
agences_raw = spark.table(TBL_AGENCE)
articles_raw = spark.table(TBL_ARTICLES)
fac_raw = spark.table(TBL_FACTURATION)

silver_ventes = materialize_table(build_silver_ventes(train_raw), TMP_SILVER)
articles_enc = build_articles_encoded(articles_raw)
agences_enc = build_agences_encoded(agences_raw)
fac_lagged = build_facturation_lagged(fac_raw)
panel = materialize_table(build_silver_panel(silver_ventes, test_raw), TMP_PANEL)

print(f"Silver panel rows: {panel.count():,}")


## 2. Pipe-1 Gold Feature Builder


In [ ]:
def build_features_from_y(panel_with_y, articles_enc, agences_enc, fac):
    df = panel_with_y
    pair_order = Window.partitionBy(*PAIR_KEYS).orderBy("week_id")

    for n in LAGS_ALL:
        df = df.withColumn(f"lag_{n}", F.lag("y", n).over(pair_order))

    def _lookback(n):
        return (
            Window.partitionBy(*PAIR_KEYS)
            .orderBy("week_id")
            .rowsBetween(-n, -1)
        )

    for n in ROLLING_WINDOWS:
        w = _lookback(n)
        df = (
            df.withColumn(f"roll_mean_{n}", F.avg("y").over(w))
              .withColumn(f"roll_std_{n}", F.stddev("y").over(w))
        )
    for n in ROLLING_MEDIAN_WINDOWS:
        df = df.withColumn(
            f"roll_median_{n}",
            F.expr("percentile_approx(y, 0.5)").over(_lookback(n)),
        )

    df = df.withColumn(
        "_y_is_zero",
        F.when(F.col("y").isNull(), F.lit(None).cast("double"))
         .when(F.col("y") == 0, F.lit(1.0))
         .otherwise(F.lit(0.0)),
    )
    df = (
        df
        .withColumn("zero_rate_26", F.avg("_y_is_zero").over(_lookback(26)))
        .withColumn("zero_rate_52", F.avg("_y_is_zero").over(_lookback(52)))
        .withColumn(
            "pair_zero_rate_expanding",
            F.avg("_y_is_zero").over(
                Window.partitionBy(*PAIR_KEYS)
                .orderBy("week_id")
                .rowsBetween(Window.unboundedPreceding, -1)
            ),
        )
    )

    recent_w = Window.partitionBy(*PAIR_KEYS).orderBy("week_id").rowsBetween(-4, -1)
    prev_w = Window.partitionBy(*PAIR_KEYS).orderBy("week_id").rowsBetween(-8, -5)
    df = (
        df
        .withColumn("_mean_recent4", F.avg("y").over(recent_w))
        .withColumn("_mean_prev4", F.avg("y").over(prev_w))
        .withColumn(
            "trend_8",
            F.when(F.col("_mean_prev4").isNull(), F.lit(None).cast("double"))
             .otherwise(
                 F.least(
                     F.greatest(
                         (F.col("_mean_recent4") - F.col("_mean_prev4"))
                         / (F.col("_mean_prev4") + F.lit(1.0)),
                         F.lit(-5.0),
                     ),
                     F.lit(5.0),
                 )
             ),
        )
        .withColumn(
            "yoy_ratio",
            F.when(
                F.col("lag_104").isNull() | (F.col("lag_104") == 0),
                F.lit(None).cast("double"),
            ).otherwise(F.col("lag_52") / F.col("lag_104")),
        )
    )

    pair_exp = (
        Window.partitionBy(*PAIR_KEYS)
        .orderBy("week_id")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    df = (
        df
        .withColumn("pair_mean", F.avg("y").over(pair_exp))
        .withColumn("pair_median", F.expr("percentile_approx(y, 0.5)").over(pair_exp))
        .withColumn("pair_max", F.max("y").over(pair_exp))
        .withColumn("pair_count", F.count("y").over(pair_exp))
        .withColumn("_pair_std", F.stddev("y").over(pair_exp))
        .withColumn(
            "pair_cv",
            F.when((F.col("pair_mean").isNull()) | (F.col("pair_mean") == 0), None)
             .otherwise(F.col("_pair_std") / (F.col("pair_mean") + F.lit(1e-6))),
        )
        .withColumn(
            "ratio_n1_vs_mean",
            F.when((F.col("pair_mean").isNull()) | (F.col("pair_mean") == 0), None)
             .otherwise(F.col("lag_52") / F.col("pair_mean")),
        )
        .withColumn("n_active_weeks", F.sum((F.col("y") > 0).cast("double")).over(pair_exp))
    )

    season_w = (
        Window.partitionBy(*PAIR_KEYS, "num_sem")
        .orderBy("annee")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    df = (
        df
        .withColumn("sem_mean", F.avg("y").over(season_w))
        .withColumn("sem_max", F.max("y").over(season_w))
        .withColumn("sem_median", F.expr("percentile_approx(y, 0.5)").over(season_w))
    )

    ag_w = Window.partitionBy("code_agence").orderBy("week_id").rowsBetween(Window.unboundedPreceding, -1)
    art_w = Window.partitionBy("code_article").orderBy("week_id").rowsBetween(Window.unboundedPreceding, -1)
    df = (
        df
        .withColumn("agence_mean", F.avg("y").over(ag_w))
        .withColumn("agence_median", F.expr("percentile_approx(y, 0.5)").over(ag_w))
        .withColumn("article_mean", F.avg("y").over(art_w))
        .withColumn("article_median", F.expr("percentile_approx(y, 0.5)").over(art_w))
    )

    two_pi = F.lit(2 * math.pi)
    df = (
        df
        .withColumn("sin_sem", F.sin(two_pi * F.col("num_sem") / F.lit(52.0)))
        .withColumn("cos_sem", F.cos(two_pi * F.col("num_sem") / F.lit(52.0)))
        .withColumn("is_summer_trough", ((F.col("num_sem") >= 30) & (F.col("num_sem") <= 35)).cast("tinyint"))
        .withColumn("is_xmas_trough", ((F.col("num_sem") >= 50) | (F.col("num_sem") == 1)).cast("tinyint"))
    )

    df = df.join(articles_enc, PAIR_KEYS, "left").join(agences_enc, "code_agence", "left")

    df = (
        df
        .withColumn(
            "_join_mois",
            F.least(F.lit(12), F.greatest(F.lit(1), F.ceil(F.col("num_sem") / F.lit(4.333)))),
        )
        .withColumn("_join_annee", F.col("annee"))
        .join(fac, ["code_agence", "code_article", "_join_annee", "_join_mois"], "left")
        .drop("_join_annee", "_join_mois")
    )

    base_cols = [
        "semaine", "week_id", "code_agence", "code_article",
        "quantite", "quantite_raw", "quantite_smooth",
        "is_anomaly", "is_capped", "is_dead_pair",
    ]
    for c in FEATURES:
        if c not in df.columns:
            df = df.withColumn(c, F.lit(None).cast("double"))

    return df.select(*base_cols, *FEATURES)


def panel_with_history_y(panel, history_end_week_id, pred_sdf=None):
    if pred_sdf is not None:
        base = panel.join(pred_sdf, ["semaine", "code_agence", "code_article"], "left")
    else:
        base = panel.withColumn("_iter_pred", F.lit(None).cast("double"))

    return base.withColumn(
        "y",
        F.when(F.col("_iter_pred").isNotNull(), F.col("_iter_pred"))
         .when(F.col("week_id") <= F.lit(history_end_week_id), F.col("quantite").cast("double"))
         .otherwise(F.lit(None).cast("double")),
    )


def build_static_features(history_end_week_id, start_week_id, end_week_id):
    return (
        build_features_from_y(
            panel_with_history_y(panel, history_end_week_id),
            articles_enc,
            agences_enc,
            fac_lagged,
        )
        .filter((F.col("week_id") >= F.lit(start_week_id)) & (F.col("week_id") <= F.lit(end_week_id)))
    )


## 3. Static Train Features for LightGBM Training
The model is trained only on the train period with complete real lag data.
Validation/test rows are built later by the iterative scorer, where lag_1,
lag_2 and the rolling features can consume previous predictions.


In [ ]:
train_features_sdf = materialize_table(build_static_features(TRAIN_END_WEEK_ID, 0, TRAIN_END_WEEK_ID), TMP_TRAIN_FEATURES)

cols_needed = ["semaine", "code_agence", "code_article", "week_id", "quantite", "is_dead_pair"] + FEATURES
train_pd = train_features_sdf.select(*cols_needed).toPandas()

print(f"Train rows: {len(train_pd):,}")


In [ ]:
# COMMAND ----------

# Build compact pandas state for the fast iterative scorer.
silver_pd = silver_ventes.toPandas()

test_pd = _add_time_columns(test_raw).toPandas()
test_pd["quantite_raw"] = np.nan
test_pd["quantite"] = np.nan
test_pd["quantite_smooth"] = np.nan
test_pd["is_anomaly"] = 0
test_pd["is_capped"] = 0
test_pd["is_dead_pair"] = 0

base_cols = [
    "semaine", "annee", "num_sem", "week_id", "code_agence", "code_article",
    "quantite_raw", "quantite", "quantite_smooth",
    "is_anomaly", "is_capped", "is_dead_pair",
]
panel_pd = pd.concat([silver_pd[base_cols], test_pd[base_cols]], ignore_index=True)
panel_pd = panel_pd.sort_values(["week_id", "code_agence", "code_article"]).reset_index(drop=True)

articles_pd = articles_enc.toPandas()
agences_pd = agences_enc.toPandas()
fac_pd = fac_lagged.toPandas()

for df in [panel_pd, articles_pd, agences_pd, fac_pd]:
    for c in ["code_agence", "code_article"]:
        if c in df.columns:
            df[c] = df[c].astype("int64")

print(f"Pandas panel rows: {len(panel_pd):,}")


def _mean(vals):
    return float(np.mean(vals)) if len(vals) else np.nan


def _median(vals):
    return float(np.median(vals)) if len(vals) else np.nan


def _max(vals):
    return float(np.max(vals)) if len(vals) else np.nan


def _std(vals):
    return float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan


def _zero_rate(vals):
    return float(np.mean(np.asarray(vals) == 0.0)) if len(vals) else np.nan


def _ratio(num, den):
    if pd.isna(num) or pd.isna(den) or den == 0:
        return np.nan
    return float(num / den)


def horizon_rows(start_week_id, end_week_id):
    cols = ["semaine", "annee", "num_sem", "week_id", "code_agence", "code_article", "quantite", "is_dead_pair"]
    out = panel_pd.loc[(panel_pd["week_id"] >= start_week_id) & (panel_pd["week_id"] <= end_week_id), cols].copy()
    out["_join_mois"] = np.ceil(out["num_sem"] / 4.333).clip(1, 12).astype("int64")
    out["_join_annee"] = out["annee"].astype("int64")
    return out.sort_values(["week_id", "code_agence", "code_article"]).reset_index(drop=True)


def prepare_state(history_end_week_id):
    hist = panel_pd.loc[(panel_pd["week_id"] <= history_end_week_id) & panel_pd["quantite"].notna()].copy()
    hist = hist.sort_values(["week_id", "code_agence", "code_article"])
    pair_hist, pair_sem_hist, agency_hist, article_hist, pair_sum = {}, {}, {}, {}, {}
    for row in hist.itertuples(index=False):
        ag = int(row.code_agence)
        art = int(row.code_article)
        sem = int(row.num_sem)
        pair = (ag, art)
        y = float(row.quantite)
        pair_hist.setdefault(pair, []).append(y)
        pair_sem_hist.setdefault((ag, art, sem), []).append(y)
        agency_hist.setdefault(ag, []).append(y)
        article_hist.setdefault(art, []).append(y)
        pair_sum[pair] = pair_sum.get(pair, 0.0) + y
    return {"pair": pair_hist, "pair_sem": pair_sem_hist, "agency": agency_hist, "article": article_hist, "pair_sum": pair_sum}


def clone_state(state):
    return {
        "pair": {k: v.copy() for k, v in state["pair"].items()},
        "pair_sem": {k: v.copy() for k, v in state["pair_sem"].items()},
        "agency": {k: v.copy() for k, v in state["agency"].items()},
        "article": {k: v.copy() for k, v in state["article"].items()},
        "pair_sum": state["pair_sum"].copy(),
    }


base_state_val = prepare_state(TRAIN_END_WEEK_ID)
base_state_test = prepare_state(VAL_END_WEEK_ID)
base_state_final = prepare_state(INTERNAL_TEST_END_WEEK_ID)
val_horizon = horizon_rows(VAL_START_WEEK_ID, VAL_END_WEEK_ID)
internal_test_horizon = horizon_rows(INTERNAL_TEST_START_WEEK_ID, INTERNAL_TEST_END_WEEK_ID)
final_horizon = horizon_rows(FINAL_INFERENCE_START_WEEK_ID, FINAL_INFERENCE_END_WEEK_ID)
print("Fast iterative states ready.")


## 4. Same Two-Stage LightGBM Model as `03_train_model.py`


In [ ]:
def build_xy(df: pd.DataFrame):
    X = df[FEATURES].copy()
    for c in FEATURES_CATEGORICAL:
        if c in X.columns:
            X[c] = X[c].astype("category")
    y = df["quantite"].astype(float).values
    is_zero = (y == 0).astype(int)
    return X, y, is_zero


X_tr, y_tr, z_tr = build_xy(train_pd)

print(f"Zero rate train: {z_tr.mean():.3f}")


In [ ]:
RUN_ID = None
with mlflow.start_run(run_name="fast_iterative_notebook") as run:
    RUN_ID = run.info.run_id
    mlflow.log_params({
        "n_features": len(FEATURES),
        "train_rows": len(train_pd),
        "train_end": TRAIN_END_WEEK_ID,
        "val_start": VAL_START_WEEK_ID,
        "val_end": VAL_END_WEEK_ID,
        "internal_test_start": INTERNAL_TEST_START_WEEK_ID,
        "internal_test_end": INTERNAL_TEST_END_WEEK_ID,
        "materialize_intermediate_delta": MATERIALIZE_INTERMEDIATE_DELTA,
        "iter_feed_rounded": ITER_FEED_ROUNDED,
    })

    dtrain_z = lgb.Dataset(X_tr, label=z_tr, categorical_feature=FEATURES_CATEGORICAL)
    model_zero = lgb.train(
        LGB_PARAMS_ZERO,
        dtrain_z,
        num_boost_round=LGB_NUM_ROUNDS_ZERO,
        valid_sets=[dtrain_z],
        valid_names=["train"],
        callbacks=[lgb.log_evaluation(period=100)],
    )

    nz = y_tr > 0
    X_tr_nz = X_tr.loc[nz].reset_index(drop=True)
    y_tr_nz = y_tr[nz]

    qty_params = dict(LGB_PARAMS_QTY)
    qty_params["objective"] = "regression_l1"
    qty_params["metric"] = "None"

    dtrain_q = lgb.Dataset(X_tr_nz, label=y_tr_nz, categorical_feature=FEATURES_CATEGORICAL)
    model_qty = lgb.train(
        qty_params,
        dtrain_q,
        num_boost_round=LGB_NUM_ROUNDS_QTY,
        valid_sets=[dtrain_q],
        valid_names=["train"],
        feval=wape_lgb_feval,
        callbacks=[lgb.log_evaluation(period=100)],
    )

    mlflow.log_param("zero_total_iterations", model_zero.current_iteration())
    mlflow.log_param("qty_total_iterations", model_qty.current_iteration())

    X_sig = X_tr.head(5).copy()
    for c in FEATURES_CATEGORICAL:
        if c in X_sig.columns:
            X_sig[c] = X_sig[c].astype(int)
    mlflow.lightgbm.log_model(
        model_zero,
        artifact_path="iter_zero_classifier",
        signature=infer_signature(X_sig, model_zero.predict(X_tr.head(5))),
        input_example=X_sig.head(1),
    )

    X_q_sig = X_tr_nz.head(5).copy()
    for c in FEATURES_CATEGORICAL:
        if c in X_q_sig.columns:
            X_q_sig[c] = X_q_sig[c].astype(int)
    mlflow.lightgbm.log_model(
        model_qty,
        artifact_path="iter_qty_regressor",
        signature=infer_signature(X_q_sig, model_qty.predict(X_tr_nz.head(5))),
        input_example=X_q_sig.head(1),
    )

print(f"Trained zero iterations: {model_zero.current_iteration()}")
print(f"Trained qty iterations : {model_qty.current_iteration()}")


## 5. Fast Pandas/Numpy Iterative Scorer

This replaces the original Spark-per-week scorer. Each week is now a pandas/numpy feature build over the horizon rows, then the predictions are fed back into the in-memory state before the next week.


In [ ]:
def feature_rows_for_week(week_df, state):
    ag_stats = {ag: (_mean(vals), _median(vals)) for ag, vals in state["agency"].items()}
    art_stats = {art: (_mean(vals), _median(vals)) for art, vals in state["article"].items()}
    rows = []

    for r in week_df.itertuples(index=False):
        ag = int(r.code_agence)
        art = int(r.code_article)
        sem = int(r.num_sem)
        pair = (ag, art)
        vals = state["pair"].get(pair, [])
        sem_vals = state["pair_sem"].get((ag, art, sem), [])

        def lag(n):
            return vals[-n] if len(vals) >= n else np.nan

        def tail(n):
            return vals[-n:] if len(vals) else []

        lag_52 = lag(52)
        lag_104 = lag(104)
        pair_mean = _mean(vals)
        pair_std = _std(vals)
        recent4 = tail(4)
        prev4 = vals[-8:-4] if len(vals) >= 5 else []
        mean_recent4 = _mean(recent4)
        mean_prev4 = _mean(prev4)

        trend_8 = np.nan
        if not pd.isna(mean_prev4):
            trend_8 = min(max((mean_recent4 - mean_prev4) / (mean_prev4 + 1.0), -5.0), 5.0)

        ag_mean, ag_median = ag_stats.get(ag, (np.nan, np.nan))
        art_mean, art_median = art_stats.get(art, (np.nan, np.nan))

        row = {
            "semaine": r.semaine, "week_id": int(r.week_id), "annee": int(r.annee), "num_sem": sem,
            "code_agence": ag, "code_article": art, "quantite": r.quantite,
            "is_dead_pair": 1 if state["pair_sum"].get(pair, 0.0) == 0 else 0,
            "lag_1": lag(1), "lag_2": lag(2), "lag_4": lag(4), "lag_8": lag(8), "lag_13": lag(13),
            "lag_26": lag(26), "lag_52": lag_52, "lag_104": lag_104,
            "zero_rate_26": _zero_rate(tail(26)), "zero_rate_52": _zero_rate(tail(52)),
            "pair_zero_rate_expanding": _zero_rate(vals),
            "trend_8": trend_8,
            "ratio_n1_vs_mean": _ratio(lag_52, pair_mean),
            "yoy_ratio": _ratio(lag_52, lag_104),
            "pair_mean": pair_mean, "pair_median": _median(vals), "pair_max": _max(vals),
            "pair_count": len(vals),
            "pair_cv": np.nan if pd.isna(pair_mean) or pair_mean == 0 else pair_std / (pair_mean + 1e-6),
            "sem_mean": _mean(sem_vals), "sem_max": _max(sem_vals), "sem_median": _median(sem_vals),
            "agence_mean": ag_mean, "agence_median": ag_median,
            "article_mean": art_mean, "article_median": art_median,
            "n_active_weeks": float(np.sum(np.asarray(vals) > 0)) if len(vals) else 0.0,
            "annee": int(r.annee), "num_sem": sem,
            "sin_sem": math.sin(2 * math.pi * sem / 52.0),
            "cos_sem": math.cos(2 * math.pi * sem / 52.0),
            "is_summer_trough": 1 if 30 <= sem <= 35 else 0,
            "is_xmas_trough": 1 if sem >= 50 or sem == 1 else 0,
        }
        for n in ROLLING_WINDOWS:
            tv = tail(n)
            row[f"roll_mean_{n}"] = _mean(tv)
            row[f"roll_std_{n}"] = _std(tv)
        for n in ROLLING_MEDIAN_WINDOWS:
            row[f"roll_median_{n}"] = _median(tail(n))
        rows.append(row)

    feat = pd.DataFrame(rows)
    feat = feat.merge(articles_pd, on=["code_agence", "code_article"], how="left")
    feat = feat.merge(agences_pd, on="code_agence", how="left")

    fac_cols = ["code_agence", "code_article", "_join_annee", "_join_mois"]
    fac_join = week_df[fac_cols].merge(fac_pd, on=fac_cols, how="left")
    for c in ["fac_prix_unit", "fac_pct_pro", "fac_nb_chantiers", "fac_nb_achats"]:
        feat[c] = fac_join[c].values if c in fac_join.columns else np.nan

    for c in FEATURES:
        if c not in feat.columns:
            feat[c] = np.nan
    return feat


def append_predictions_to_state(state, scored_week):
    for r in scored_week.itertuples(index=False):
        ag = int(r.code_agence)
        art = int(r.code_article)
        sem = int(r.num_sem)
        pair = (ag, art)
        y_feed = float(r.prediction_int if ITER_FEED_ROUNDED else r.prediction)
        state["pair"].setdefault(pair, []).append(y_feed)
        state["pair_sem"].setdefault((ag, art, sem), []).append(y_feed)
        state["agency"].setdefault(ag, []).append(y_feed)
        state["article"].setdefault(art, []).append(y_feed)
        state["pair_sum"][pair] = state["pair_sum"].get(pair, 0.0) + y_feed


def score_horizon_iterative_fast(base_state, horizon_df, zero_iter, qty_iter, threshold, label):
    state = clone_state(base_state)
    out_parts = []
    for week_id in sorted(horizon_df["week_id"].unique()):
        week_df = horizon_df[horizon_df["week_id"] == week_id].copy()
        feat = feature_rows_for_week(week_df, state)
        X = feat[FEATURES].copy()
        for c in FEATURES_CATEGORICAL:
            if c in X.columns:
                X[c] = X[c].fillna(-1).astype("int64").astype("category")
        p_zero = model_zero.predict(X, num_iteration=int(zero_iter))
        qty = np.clip(model_qty.predict(X, num_iteration=int(qty_iter)), 0.0, None)
        pred = np.where(p_zero > threshold, 0.0, qty)
        pred = np.where(feat["is_dead_pair"].values == 1, 0.0, pred)
        scored = feat[["semaine", "week_id", "annee", "num_sem", "code_agence", "code_article", "quantite", "is_dead_pair"]].copy()
        scored["p_zero"] = p_zero
        scored["qty_pred"] = qty
        scored["prediction"] = pred
        scored["prediction_int"] = np.clip(np.round(pred), 0, None).astype(np.int64)
        scored["split"] = label
        out_parts.append(scored)
        append_predictions_to_state(state, scored)
    return pd.concat(out_parts, ignore_index=True)


def candidate_iters(total_iter, step, max_candidates):
    vals = {int(total_iter)}
    vals.update(range(step, int(total_iter) + 1, step))
    vals = sorted(v for v in vals if v > 0)
    if len(vals) > max_candidates:
        idx = np.linspace(0, len(vals) - 1, max_candidates).round().astype(int)
        vals = sorted(set(vals[i] for i in idx) | {int(total_iter)})
    return vals


## 6. Iterative Validation Model Selection

The validation WAPE used for model selection is now iterative: predictions from previous validation weeks feed the following week's lags/rolling features. This is a post-hoc `num_iteration` selection rather than native LightGBM early stopping because the validation matrix itself depends on recursive predictions.


In [ ]:
ZERO_ITER_CANDIDATES = candidate_iters(model_zero.current_iteration(), ZERO_ITER_CANDIDATE_STEP, MAX_ZERO_ITER_CANDIDATES)
QTY_ITER_CANDIDATES = candidate_iters(model_qty.current_iteration(), QTY_ITER_CANDIDATE_STEP, MAX_QTY_ITER_CANDIDATES)

print("ZERO_ITER_CANDIDATES:", ZERO_ITER_CANDIDATES)
print("QTY_ITER_CANDIDATES :", QTY_ITER_CANDIDATES)
print("THRESHOLDS          :", THRESHOLD_GRID_MODEL_SELECT)

selection_rows = []


def evaluate_iterative_candidate(zi, qi, thr, stage):
    pred_val = score_horizon_iterative_fast(base_state_val, val_horizon, zi, qi, thr, f"validation_select_{stage}")
    wape = wape_numpy(pred_val["quantite"].values, pred_val["prediction"].values)
    row = {"stage": stage, "zero_iter": int(zi), "qty_iter": int(qi), "threshold": float(thr), "val_wape": wape}
    selection_rows.append(row)
    print(f"{stage:>10s} | zi={int(zi):4d} qi={int(qi):4d} thr={thr:.2f} -> iterative val WAPE={wape:.5f}")
    return row


if RUN_ITERATIVE_MODEL_SELECTION:
    current_zi = int(model_zero.current_iteration())
    current_qi = int(model_qty.current_iteration())
    current_thr = float(DEFAULT_ZERO_THRESHOLD)

    if MODEL_SELECTION_STRATEGY == "exhaustive":
        for zi in ZERO_ITER_CANDIDATES:
            for qi in QTY_ITER_CANDIDATES:
                for thr in THRESHOLD_GRID_MODEL_SELECT:
                    evaluate_iterative_candidate(zi, qi, thr, "exhaustive")
    elif MODEL_SELECTION_STRATEGY == "coordinate":
        for pass_id in range(MODEL_SELECTION_COORDINATE_PASSES):
            qty_rows = [evaluate_iterative_candidate(current_zi, qi, current_thr, f"qty_p{pass_id + 1}") for qi in QTY_ITER_CANDIDATES]
            current_qi = int(min(qty_rows, key=lambda r: r["val_wape"])["qty_iter"])

            zero_rows = [evaluate_iterative_candidate(zi, current_qi, current_thr, f"zero_p{pass_id + 1}") for zi in ZERO_ITER_CANDIDATES]
            current_zi = int(min(zero_rows, key=lambda r: r["val_wape"])["zero_iter"])

            thr_rows = [evaluate_iterative_candidate(current_zi, current_qi, thr, f"thr_p{pass_id + 1}") for thr in THRESHOLD_GRID_MODEL_SELECT]
            current_thr = float(min(thr_rows, key=lambda r: r["val_wape"])["threshold"])
    else:
        raise ValueError(f"Unknown MODEL_SELECTION_STRATEGY: {MODEL_SELECTION_STRATEGY}")

    selection_df = pd.DataFrame(selection_rows).sort_values("val_wape")
else:
    BEST_ZERO_ITER = int(MANUAL_ZERO_ITER or model_zero.current_iteration())
    BEST_QTY_ITER = int(MANUAL_QTY_ITER or model_qty.current_iteration())
    BEST_THRESHOLD = float(MANUAL_ZERO_THRESHOLD)
    manual_row = evaluate_iterative_candidate(BEST_ZERO_ITER, BEST_QTY_ITER, BEST_THRESHOLD, "manual")
    selection_df = pd.DataFrame([manual_row]).sort_values("val_wape")

display(selection_df)
best = selection_df.iloc[0]
BEST_ZERO_ITER = int(best["zero_iter"])
BEST_QTY_ITER = int(best["qty_iter"])
BEST_THRESHOLD = float(best["threshold"])
BEST_VAL_WAPE = float(best["val_wape"])
print(f"BEST zero_iter={BEST_ZERO_ITER}, qty_iter={BEST_QTY_ITER}, threshold={BEST_THRESHOLD}, val_wape={BEST_VAL_WAPE:.5f}")

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_param("iterative_model_selection_strategy", MODEL_SELECTION_STRATEGY if RUN_ITERATIVE_MODEL_SELECTION else "manual")
    mlflow.log_param("best_zero_iter_iterative", BEST_ZERO_ITER)
    mlflow.log_param("best_qty_iter_iterative", BEST_QTY_ITER)
    mlflow.log_param("best_threshold_iterative", BEST_THRESHOLD)
    mlflow.log_metric("best_val_wape_iterative", BEST_VAL_WAPE)


## 7. Iterative Validation and Internal Test


In [ ]:
val_iter = score_horizon_iterative_fast(base_state_val, val_horizon, BEST_ZERO_ITER, BEST_QTY_ITER, BEST_THRESHOLD, "validation")
val_wape = wape_numpy(val_iter["quantite"].values, val_iter["prediction"].values)
print(f"Validation iterative WAPE: {val_wape:.5f}")

internal_test_iter = score_horizon_iterative_fast(base_state_test, internal_test_horizon, BEST_ZERO_ITER, BEST_QTY_ITER, BEST_THRESHOLD, "internal_test")
internal_test_wape = wape_numpy(internal_test_iter["quantite"].values, internal_test_iter["prediction"].values)
print(f"Internal test iterative WAPE: {internal_test_wape:.5f}")

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metric("val_wape_iterative_final", val_wape)
    mlflow.log_metric("internal_test_wape_iterative", internal_test_wape)

display(pd.DataFrame([
    {"split": "validation", "wape": val_wape},
    {"split": "internal_test", "wape": internal_test_wape},
]))


In [ ]:
def write_scored(df, table_name):
    out = df[["semaine", "code_agence", "code_article", "quantite", "p_zero", "qty_pred", "prediction", "prediction_int"]].copy()
    sdf = (
        spark.createDataFrame(out)
        .withColumn("code_agence", F.col("code_agence").cast(LongType()))
        .withColumn("code_article", F.col("code_article").cast(LongType()))
        .withColumn("prediction_int", F.col("prediction_int").cast(LongType()))
    )
    (
        sdf.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"Wrote {sdf.count():,} rows to {table_name}")

write_scored(val_iter, OUT_VAL_ITER)
write_scored(internal_test_iter, OUT_INTERNAL_TEST_ITER)


## 8. Final Iterative Leaderboard-Horizon Inference


In [ ]:
final_iter = score_horizon_iterative_fast(base_state_final, final_horizon, BEST_ZERO_ITER, BEST_QTY_ITER, BEST_THRESHOLD, "final_inference")
submission = final_iter[["semaine", "code_agence", "code_article", "prediction_int"]].copy()
submission = submission.rename(columns={"prediction_int": "quantite"})
submission["quantite"] = np.clip(submission["quantite"], 0, None).astype(np.int64)

submission_sdf = (
    spark.createDataFrame(submission)
    .withColumn("code_agence", F.col("code_agence").cast(LongType()))
    .withColumn("code_article", F.col("code_article").cast(LongType()))
    .withColumn("quantite", F.col("quantite").cast(LongType()))
)
(
    submission_sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_FINAL_ITER)
)

n_rows = submission_sdf.count()
n_pos = submission_sdf.filter(F.col("quantite") > 0).count()
print(f"Wrote iterative final submission candidate: {OUT_FINAL_ITER}")
print(f"Rows: {n_rows:,}; positive predictions: {n_pos:,}")

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_param("iterative_final_table", OUT_FINAL_ITER)
    mlflow.log_metric("iterative_final_rows", n_rows)
    mlflow.log_metric("iterative_final_positive_predictions", n_pos)
